In [1]:
import numpy as np
from copy import deepcopy
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
from tensorflow.keras import layers,regularizers,metrics,optimizers
import random
import pandas as pd
from scipy.linalg import sqrtm
import pickle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import math
from collections import deque
import tarfile
from PIL import Image
from scipy.io import loadmat
import io
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

KeyboardInterrupt: 

In [ ]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [ ]:
UTKFace="C://Users//user//noise//class-2//datas//UTKFace.tar.gz"
crop_part1="C://Users//user//noise//class-2//datas//crop_part1.tar.gz"

In [ ]:
def load_utkface_no_landmarks(tar_paths, target_size=(128,128)):
    X = []
    age_list, gender_list, race_list = [], [], []

    for tar_path in tar_paths:
        tar = tarfile.open(tar_path, "r:*")

        for member in tar.getmembers():
            if not member.isfile():
                continue
            if not (member.name.endswith(".jpg") or member.name.endswith(".png")):
                continue

            filename = member.name.split("/")[-1]
            parts = filename.split("_")

            # standard format: age_gender_race_date.jpg
            if len(parts) < 4:
                continue

            try:
                age = int(parts[0])
                gender = int(parts[1])
                race = int(parts[2])
            except:
                continue  # skip malformed filenames

            # read image
            img_bytes = tar.extractfile(member).read()
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

            img = img.resize(target_size)
            img = np.array(img, dtype=np.float32) / 255.0

            X.append(img)
            age_list.append(age)
            gender_list.append(gender)
            race_list.append(race)

        tar.close()

    return (
        np.array(X, dtype=np.float32),
        np.array(age_list),
        np.array(gender_list),
        np.array(race_list)
    )

In [ ]:
def downsample_tf(X):
    """
    X: tensor or numpy array of shape (N,128,128,3)
    """
    X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
    X_small = tf.image.resize(X_tf, (32, 32), method='area')
    return X_small.numpy()

In [ ]:
SAVE_PATH = "utkface_12k_split.npz"   # file to save 10000+2000 data

def age_to_bins(age, bin_size=5):
    """Map age to bins, e.g., 0-4 → 0, 5-9 → 1, ..."""
    return (age // bin_size).astype(np.int32)

def stratified_sample(X, age, n_train, n_test, random_state=42):
    """
    Stratified sampling by age to preserve distribution in train/test.
    """

    # 1. bin ages
    bins = age_to_bins(age)

    # 2. stratified sampling
    sss = StratifiedShuffleSplit(
        n_splits=1,
        train_size=n_train,
        test_size=n_test,
        random_state=random_state
    )

    idx_train, idx_test = next(sss.split(X, bins))

    return X[idx_train], X[idx_test], age[idx_train], age[idx_test]

def save_dataset(x_train, y_train, x_test, y_test, path=SAVE_PATH):
    np.savez_compressed(
        path,
        x_train=x_train,
        y_train=y_train,
        x_test=x_test,
        y_test=y_test
    )
    print(f"✔ Data saved to: {path}")

def load_dataset_if_exists(path=SAVE_PATH):
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return data["x_train"], data["y_train"], data["x_test"], data["y_test"]
    return None

def get_utkface_dataset(downsample_func, random_state=42):
    """
    X: raw images (N,H,W,C)
    age: age labels (N,)
    downsample_func: your downsample_tf or opencv version
    """

    # 1. check cache
    cached = load_dataset_if_exists()
    if cached is not None:
        return cached

    # 2. no cache -> stratified sample 10000 + 2000
    print("⚠ No cache found, generating dataset via stratified sampling ...")

    X, age, gender, race = load_utkface_no_landmarks(
        [UTKFace, crop_part1],     # your path or variable
        target_size=(128,128))
    age = (age+1)/120

    x_train, x_test, y_train, y_test = stratified_sample(
        X, age,
        n_train=20000,
        n_test=2000,
        random_state=random_state
    )

    print("✔ Sampling complete, downsampling to 64x64 ...")
    x_train = downsample_func(x_train)
    x_test = downsample_func(x_test)

    # 3. save
    save_dataset(x_train, y_train, x_test, y_test)

    return x_train, y_train, x_test, y_test

In [ ]:
x_train, y_train, x_test, y_test = get_utkface_dataset(downsample_tf)